# 03. Transformations, Interpolation, And Screw Motion

This notebook uses Pinocchio `SE3` objects directly. The goal is to build intuition for composing rigid transforms, changing reference frames, and comparing different interpolation strategies.

## What you should remember
- `pin.SE3(R, p)` stores a rigid transform.
- Composition uses `*`, and inverses use `.inverse()`.
- `pin.log` / `pin.exp` move between `SE3` and `se(3)` (lie algebra) -> errors should be taken in se(3) (Vector Space).
- Translational interpolation in `R3`, rotational interpolation in `SO3`, and interpolation directly in `SE3` are related but not identical.
- The exact same `SE3` pose can drive frames, primitive meshes, or a more interesting mesh such as the Stanford bunny.

In [ ]:
%%capture
!pip install pin viser robot_descriptions numpy scipy matplotlib trimesh

import site
site.main()

In [ ]:
%%capture
!wget -O viz.py https://raw.githubusercontent.com/Atarilab/colab_utils/refs/heads/main/viz.py
import viz

If the next cell gives an error restart the session to load the libraries (ctrl + m + .) or click runtime -> restart session. Then rerun the second and third codeblocks (do not rerun the first block!).

In [ ]:
from functools import partial
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pinocchio as pin
import trimesh

from viz import PinNotebookViz, add_mesh_handle, create_server, update_object_pose

server, share_url = create_server()
notebook_viz = PinNotebookViz(server)
add_mesh_handle = partial(add_mesh_handle, server)

print(f"Open the visualizer here: {share_url}")

## Generate a couple of random poses
Instead of hard-coding the frames, we build a small helper so we can repeatedly test the same ideas on random examples.

In [ ]:
T_world_start = pin.SE3.Random()
T_world_goal = pin.SE3.Random()

print("Start pose:")
print(T_world_start)
print()
print("Goal pose:")
print(T_world_goal)

notebook_viz.clear_frames()
notebook_viz.show_frame("world", pin.SE3.Identity(), axes_length=0.20)
notebook_viz.show_frame("start", T_world_start)
notebook_viz.show_frame("goal", T_world_goal)

## Relative transforms
If you know two absolute poses, the relative transform is just another `SE3` product.

In [ ]:
T_start_goal = T_world_start.inverse() * T_world_goal

print("Relative transform from start to goal:")
print(T_start_goal)
print()
print("Twist coordinates from log(T_start_goal):")
print(pin.log(T_start_goal))

## Interpolating between frames

In [ ]:
def interpolate_r3_so3(T0, T1, alpha):
    p = (1.0 - alpha) * T0.translation + alpha * T1.translation
    R = T0.rotation @ pin.exp3(alpha * pin.log3(T0.rotation.T @ T1.rotation))
    return pin.SE3(R, p)


def interpolate_se3(T0, T1, alpha):
    return T0 * pin.exp(alpha * pin.log(T0.inverse() * T1))


notebook_viz.clear_frames()
notebook_viz.show_frame("world", pin.SE3.Identity(), axes_length=0.20)
notebook_viz.show_frame("start", T_world_start)
notebook_viz.show_frame("goal", T_world_goal)

## Interpolate translation in `R3` and rotation in `SO3`
This is the simpler construction: interpolate the position linearly in `R3`, then interpolate the orientation on `SO3`.

In [ ]:
for alpha in np.linspace(0.0, 1.0, 60):
    T_alpha = interpolate_r3_so3(T_world_start, T_world_goal, alpha)
    notebook_viz.show_frame("moving_r3_so3", T_alpha)
    time.sleep(0.03)

## Direct `SE3` interpolation and screw motion
Interpolating with `exp(alpha * log(T0^{-1} T1))` creates the one-parameter screw motion from the start pose to the goal pose.

In [ ]:
for alpha in np.linspace(0.0, 1.0, 60):
    T_alpha = interpolate_se3(T_world_start, T_world_goal, alpha)
    notebook_viz.show_frame("moving_r3_so3", T_alpha)
    time.sleep(0.03)

## Move primitives between random frames

In [ ]:
box_mesh = trimesh.creation.box(extents=(0.18, 0.10, 0.12))
box_mesh.visual.face_colors = [255, 160, 90, 190]

goal_sphere_mesh = trimesh.creation.icosphere(radius=0.07, subdivisions=2)
goal_sphere_mesh.visual.face_colors = [90, 180, 255, 140]

server.scene.reset()
notebook_viz.frame_handles = {}
notebook_viz.show_frame("world", pin.SE3.Identity(), axes_length=0.20)
notebook_viz.show_frame("start", T_world_start)
notebook_viz.show_frame("goal", T_world_goal)

box_handle = add_mesh_handle('/interp_box', box_mesh, T_world_start)
goal_sphere_handle = add_mesh_handle('/goal_sphere', goal_sphere_mesh, T_world_start)

for alpha in np.linspace(0.0, 1.0, 70):
    T_r3 = interpolate_r3_so3(T_world_start, T_world_goal, alpha)
    T_se3 = interpolate_se3(T_world_start, T_world_goal, alpha)
    update_object_pose(box_handle, T_r3)
    update_object_pose(goal_sphere_handle, T_se3)
    notebook_viz.show_frame("moving_r3_so3", T_r3)
    notebook_viz.show_frame("moving_se3", T_se3)
    time.sleep(0.03)

## Exercise: do the `SE3` interpolation from the start to the goal frame in the next cell and visualize the results.

In [ ]:
T_start = pin.SE3.Random()
T_goal = pin.SE3.Random()